In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [26]:
drive_file_path = '/content/drive/MyDrive/dataset/incident_resolution_dataset_v2_25000.csv'

try:
    df = pd.read_csv(drive_file_path)
    print(f'Successfully loaded data from: {drive_file_path}')
    display(df.head())
except FileNotFoundError:
    print(f'Error: The file at {drive_file_path} was not found. Please check the path and try again.')
except Exception as e:
    print(f'An error occurred: {e}')



Successfully loaded data from: /content/drive/MyDrive/dataset/incident_resolution_dataset_v2_25000.csv


,Incident_ID,Short_Description,Detailed_Description,Category,Priority,Department,Environment,Assigned_Group,Status,Created_At,Resolved_At,Resolution_Category,Resolution_Steps
0,INC306868,Data not saving correctly or unexpected data l...,Alex Turner in Legal accidentally deleted reco...,Database,P4,Legal,macOS Sonoma,Desktop Support,Resolved,2023-06-20 10:58:00,2023-06-25 03:02:20,Database_Issue,1. Confirmed accidental deletion scope — 14 re...
1,INC324016,Application not compatible after OS upgrade,"After upgrading to macOS Ventura, legacy app '...",Software,P4,IT,macOS Ventura,Desktop Support,Resolved,2023-08-13 14:16:00,2023-08-20 03:36:40,Software_Reinstall,1. Identified plugin API changes in new macOS ...
2,INC309668,Application not compatible after OS upgrade,"After upgrading to Windows 11, legacy app 'Ser...",Software,P3,HR,Windows 11,Database Team,Resolved,2022-03-29 07:38:00,2022-03-30 16:04:39,Software_Reinstall,1. Identified plugin API changes in new macOS ...
3,INC313640,Printer offline or producing errors on the net...,Printer in Research & Development producing ga...,Hardware,P3,Research & Development,ChromeOS,Database Team,Resolved,2022-10-30 07:54:00,2022-10-31 08:39:16,Cloud_Service_Issue,1. Stopped and restarted Print Spooler service...
4,INC314018,"Software license expired, blocking user access",Adobe CC license expired for Engineering desig...,Software,P3,Engineering,macOS Monterey,Server Team,Resolved,2022-10-19 19:14:00,2022-10-21 14:30:14,Software_Reinstall,1. Confirmed license expiry in Adobe Admin Con...


In [27]:
# Combine incident text
df['input_text'] = df['Short_Description'] + " " + df['Detailed_Description']

texts = df['input_text']
resolutions = df['Resolution_Steps']

In [28]:
# ==============================
# 2. Train Test Split
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    texts, resolutions, test_size=0.2, random_state=42
)

In [29]:
# ==============================
# 3. Tokenization
# ==============================

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 1367


In [30]:
# ==============================
# 4. Sequence Conversion
# ==============================

train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

# Decide max length
lengths = [len(x) for x in train_sequences]
max_len = int(np.percentile(lengths,95))
print("Max Length:", max_len)

X_train = pad_sequences(train_sequences, maxlen=max_len, padding='post')
X_test = pad_sequences(test_sequences, maxlen=max_len, padding='post')

Max Length: 35


In [31]:
# ==============================
# 5. Build Encoder Model
# ==============================

embedding_dim = 100
lstm_units = 128

encoder_model = Sequential([

    tf.keras.layers.Input(shape=(max_len,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    ),

    Bidirectional(
        LSTM(lstm_units, return_sequences=True)
    ),

    GlobalMaxPooling1D(),

    Dense(128, activation='relu'),

    Dense(64)   # Final embedding vector
])

encoder_model.compile(
    optimizer='adam',
    loss='mse'
)

encoder_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 35, 100)        │       136,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 35, 256)        │       234,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 412,348 (1.57 MB)

 Trainable params: 412,348 (1.57 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:

# ==============================
# 6. Train Model
# ==============================

# Dummy targets for training embeddings
y_dummy = np.random.rand(len(X_train), 64)

history = encoder_model.fit(

    X_train,
    y_dummy,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)


Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 54s 164ms/step - loss: 0.1180 - val_loss: 0.0836
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 80s 158ms/step - loss: 0.0837 - val_loss: 0.0836
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 45s 161ms/step - loss: 0.0835 - val_loss: 0.0836
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 44s 158ms/step - loss: 0.0835 - val_loss: 0.0837
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 44s 157ms/step - loss: 0.0834 - val_loss: 0.0837


In [8]:

# ==============================
# 7. Generate Embeddings
# ==============================

incident_embeddings = encoder_model.predict(X_train)


2750/2750 ━━━━━━━━━━━━━━━━━━━━ 108s 39ms/step


In [9]:
incident_embeddings

array([[0.49294624, 0.5003017 , 0.51165783, ..., 0.5132421 , 0.49833077,
        0.508798  ],
       [0.4911836 , 0.49803752, 0.50875074, ..., 0.51637876, 0.5049985 ,
        0.5137044 ],
       [0.49297577, 0.49821952, 0.5149221 , ..., 0.5109936 , 0.49763528,
        0.50773907],
       ...,
       [0.493405  , 0.49247545, 0.5109926 , ..., 0.5131345 , 0.4969462 ,
        0.5054871 ],
       [0.49382102, 0.495465  , 0.51463044, ..., 0.50845844, 0.49773455,
        0.504946  ],
       [0.49115738, 0.498462  , 0.5145291 , ..., 0.5114204 , 0.50123453,
        0.5082654 ]], dtype=float32)

In [22]:

# ==============================
# 8. Recommendation Function
# ==============================

def recommend_resolution(query, top_k=3):

    seq = tokenizer.texts_to_sequences([query])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    query_embedding = encoder_model.predict(padded)

    similarity = cosine_similarity(query_embedding, incident_embeddings)[0]
    print("Similarity", similarity.argsort()[-top_k:][::-1])

    top_indices = similarity.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:
        results.append({
            "Incident": tokenizer.sequences_to_texts([X_train[idx]])[0], # Convert sequence back to text
            "Resolution": y_train.iloc[idx],
            "Similarity": similarity[idx]
        })

    return pd.DataFrame(results)

# ==============================
# 9. Test Recommendation
# ==============================

query = "VPN connection timeout from remote office"

results = recommend_resolution(query)

print(results)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
Similarity [10116 42629 79273]
                                            Incident  \
0  outlook crashing on startup outlook crashes im...   
1  outlook crashing on startup outlook crashes im...   
2  outlook crashing on startup outlook crashes im...   

                                          Resolution  Similarity  
0  Step 1: Verify user identity.\nStep 2: Reset p...    0.999929  
1  Step 1: Verify user identity.\nStep 2: Reset p...    0.999929  
2  Step 1: Check internet connectivity.\nStep 2: ...    0.999929  


In [33]:
# ==============================
# 10. Save the Model
# ==============================

encoder_model.save('incident_encoder_model.h5')

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)